# Random forests: [AMES dataset](http://jse.amstat.org/v19n3/decock.pdf)

## Introduction

In this lab, we predict the sale price of the houses of Ames (Iowa) with a random forest, on the data that the exploration lab (`ml/ames-data-exploration`) prepared. The goal is to see what the forest brings over a single tree, and what it gives besides a prediction.

We will follow these steps:

1. write an evaluation function using cross-validation
2. compare a single tree and a forest
3. tune the forest's hyperparameters
4. read the impurity-based importance of the variables, and the forest's first tree
5. follow the out-of-bag (OOB) error as the number of trees grows
6. evaluate the chosen model on a validation set held out from the start

## Data

The files of `ames/processed/` in the [shuuchuu/datasets](https://github.com/shuuchuu/datasets/tree/main/ames) repository are the exploration lab's data, prepared as in its solution: missing values filled in, ordinal variables encoded as integers, categorical variables one-hot encoded, that is 244 columns for 1,460 houses. `train_labels.csv` holds the target, `SalePrice`.

The conclusions at the end of the exploration (removing the 4 sales of more than 4,000 sq ft, predicting the logarithm of the price) aren't applied here: a tree assumes nothing about the target's distribution, and the error thus stays in dollars.

We hold out 20% of the houses right now, a validation set that won't be used for any choice and that we'll only use at the end of the lab. Everything else is done on `train_X` and `train_y`.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames
!ls -l ames/processed/

In [ ]:
import matplotlib.pyplot as plt
import pandas
import scipy.stats
import sklearn.base
import sklearn.ensemble
import sklearn.metrics
import sklearn.model_selection
import sklearn.tree

X = pandas.read_csv("ames/processed/train_data.csv", index_col="Id")
y = pandas.read_csv("ames/processed/train_labels.csv", index_col="Id")["SalePrice"]

# Validation set: 20% of the houses, held out until the end of the lab
train_X, val_X, train_y, val_y = sklearn.model_selection.train_test_split(
  X, y, test_size=0.2, random_state=0
)
print(f"Training: {train_X.shape}, validation: {val_X.shape}")

## Evaluation

*Write a `score` function that evaluates a model with a 5-fold cross-validation on `train_X` and `train_y`, using the RMSE (root mean squared error), and returns the mean and the standard deviation of the RMSE over the 5 folds.*

In scikit-learn, the higher a score, the better: the RMSE is thus used through its opposite, the `"neg_root_mean_squared_error"` score.

**Useful documentation**

- [`sklearn.model_selection.cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [scikit-learn's scores](https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers)

In [ ]:
def score(model):
  # Your code here
  ...

### Additional hints

- `sklearn.model_selection.cross_val_score(model, train_X, train_y, cv=5, scoring="neg_root_mean_squared_error")`: an array of 5 scores, one per fold, which are negative RMSEs.
- `rmse.mean()` and `rmse.std()`: the mean and the standard deviation of a NumPy array.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
def score(model):
  rmse = -sklearn.model_selection.cross_val_score(
    model, train_X, train_y, cv=..., scoring=...
  )
  return ..., ...
```

### Solution

In [ ]:
def score(model):
  rmse = -sklearn.model_selection.cross_val_score(
    model, train_X, train_y, cv=5, scoring="neg_root_mean_squared_error"
  )
  return rmse.mean(), rmse.std()

## A tree or a forest

A deep decision tree overfits: it varies a lot from one training sample to another. The forest averages decorrelated trees to reduce this variance.

*Evaluate with `score` a single `DecisionTreeRegressor`, then a `RandomForestRegressor` of 100 trees. Which one makes the smallest error on average? Which one varies the least from one fold to another?*

**Useful documentation**

- [`sklearn.tree.DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
- [`sklearn.ensemble.RandomForestRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

In [ ]:
# Your code here

### Additional hints

- `random_state=0` in both models: identical results from one run to the next.
- `n_estimators=100`: the number of trees of the forest; `n_jobs=-1` trains them in parallel on every core.
- `score` returns a tuple `(mean, standard deviation)`.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
tree_rmse = score(sklearn.tree.DecisionTreeRegressor(random_state=0))
forest_rmse = score(...)
print(f"Single tree: ${tree_rmse[0]:,.0f} ± ${tree_rmse[1]:,.0f}")
print(f"Forest of 100 trees: ...")
```

### Solution

In [ ]:
tree_rmse = score(sklearn.tree.DecisionTreeRegressor(random_state=0))
forest_rmse = score(
  sklearn.ensemble.RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
)
print(f"Single tree: ${tree_rmse[0]:,.0f} ± ${tree_rmse[1]:,.0f}")
print(f"Forest of 100 trees: ${forest_rmse[0]:,.0f} ± ${forest_rmse[1]:,.0f}")

The single tree is off by about $41,300 on average, the forest of 100 trees by about $28,900: almost a third less error. The standard deviation, though, doesn't go down (about $5,500 for the tree, $6,200 for the forest): it mixes the model's variance, which averaging trees reduces, and the differences between the folds (some hold harder houses), which nothing reduces. Here, the differences between folds dominate.

## Optimization

*Use `RandomizedSearchCV` to tune the forest's `n_estimators`, `max_features`, `max_depth`, `min_samples_leaf` and `max_samples` parameters, with the same validation as `score` (5 folds, RMSE). Does the best configuration do better than the forest of 100 trees of the previous section?*

`RandomizedSearchCV` draws `n_iter` configurations at random. For each parameter, it takes a list of values or, above all, a probability distribution from `scipy.stats`, which explores a whole range: that's what sets it apart from `GridSearchCV`, which tries every combination of a grid. The search trains 100 forests (20 configurations, 5 folds): allow a few minutes.

**Useful documentation**

- [`sklearn.model_selection.RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
- [`scipy.stats.randint`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.randint.html)
- [`scipy.stats.uniform`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.uniform.html)

In [ ]:
# Your code here

### Additional hints

Reasonable ranges for `param_distributions`:

- `n_estimators=scipy.stats.randint(100, 300)`: an integer from 100 to 299; more trees don't overfit, but cost time.
- `max_features=["sqrt", "log2", 0.33, None]`: the number of variables drawn at each split; a float gives their fraction (0.33: the c/3 of the R package `randomForest`), `None` takes them all (scikit-learn's default for regression).
- `max_depth=[8, 16, None]`: `None` lets the trees grow all the way.
- `min_samples_leaf=scipy.stats.randint(1, 6)`: from 1 to 5 examples per leaf at least.
- `max_samples=scipy.stats.uniform(0.5, 0.5)`: the fraction of the rows drawn for each tree, between 0.5 and 1 (`uniform(loc, scale)` draws in the interval [loc, loc + scale]).

And for the search: `n_iter=20`, `cv=5`, `scoring="neg_root_mean_squared_error"`, `random_state=0` (in the search and in the forest), `n_jobs=-1`. After `fit`, `rscv.best_score_` is the best mean RMSE, negated, `rscv.best_params_` its configuration, and `rscv.best_estimator_` the forest retrained with it on the whole of `train_X`.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
rscv = sklearn.model_selection.RandomizedSearchCV(
  estimator=sklearn.ensemble.RandomForestRegressor(random_state=0),
  param_distributions=dict(
    n_estimators=scipy.stats.randint(100, 300),
    max_features=["sqrt", "log2", 0.33, None],
    max_depth=...,
    min_samples_leaf=...,
    max_samples=...,
  ),
  n_iter=20,
  scoring=...,
  cv=...,
  random_state=0,
  n_jobs=-1,
)
rscv.fit(...)

print(f"Best cross-validation RMSE: ${-rscv.best_score_:,.0f}")
print(f"Best parameters: {rscv.best_params_}")
rfr = rscv.best_estimator_
```

### Solution

In [ ]:
rscv = sklearn.model_selection.RandomizedSearchCV(
  estimator=sklearn.ensemble.RandomForestRegressor(random_state=0),
  param_distributions=dict(
    n_estimators=scipy.stats.randint(100, 300),
    max_features=["sqrt", "log2", 0.33, None],
    max_depth=[8, 16, None],
    min_samples_leaf=scipy.stats.randint(1, 6),
    max_samples=scipy.stats.uniform(0.5, 0.5),
  ),
  n_iter=20,
  scoring="neg_root_mean_squared_error",
  cv=5,
  random_state=0,
  n_jobs=-1,
)
rscv.fit(train_X, train_y)

print(f"Best cross-validation RMSE: ${-rscv.best_score_:,.0f}")
print(f"Default forest of 100 trees: ${forest_rmse[0]:,.0f}")
print(f"Best parameters: {rscv.best_params_}")

# Best model, retrained on the whole of train_X
rfr = rscv.best_estimator_

The best configuration (depth 16, a third of the variables at each split, 61% of the rows per tree, at least 2 examples per leaf, 179 trees) reaches about $28,300, against $28,900 for the default forest: barely 2% better. A random forest works well with its default settings; the search mostly serves to check that no setting changes the result much.

## Variable importance

A forest of hundreds of trees can no longer be read like a single tree: it's interpreted through the importance of its variables. A variable's impurity-based importance (`feature_importances_`) is the decrease in impurity (here, in squared error) that the splits on this variable bring, averaged over the trees. It's fast to get, but computed on the training data, and biased towards variables with many values.

*Plot the impurity-based importance of the 20 most important variables of `rfr`. Do you find the variables that the exploration found most correlated with the price, the overall quality (`OverallQual`) and the living area (`GrLivArea`)?*

**Useful documentation**

- [`pandas.Series.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html)
- [`pandas.Series.plot.barh`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.barh.html)

In [ ]:
# Your code here

### Additional hints

- `pandas.Series(rfr.feature_importances_, index=train_X.columns)`: one importance per variable, under its name; they sum to 1.
- `sort_values(ascending=False)` orders them from the most to the least important, and `.head(20)` keeps the first 20.
- `[::-1]` reverses the order, so that the most important one is at the top of the horizontal bar chart.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
importances = pandas.Series(..., index=train_X.columns)
importances = importances.sort_values(...)
importances.head(...)[::-1].plot.barh(figsize=(8, 6))
plt.show()
```

### Solution

In [ ]:
importances = pandas.Series(rfr.feature_importances_, index=train_X.columns)
importances = importances.sort_values(ascending=False)
importances.head(20)[::-1].plot.barh(figsize=(8, 6))
plt.title("Impurity-based importance of the top 20 variables")
plt.show()

## The forest's first tree

*Display the first three levels of the forest's first tree (`rfr.estimators_[0]`) with `sklearn.tree.plot_tree`. Are its first splits on the most important variables found above? Why?*

**Useful documentation**

- [`sklearn.tree.plot_tree`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html)

In [ ]:
# Your code here

### Additional hints

- `max_depth=2`: the root and two levels below it; the whole tree, with hundreds of leaves, would be unreadable.
- `feature_names=list(train_X.columns)`: the variables' names in each node, instead of their number.
- `filled=True` colors the nodes by their prediction; `plt.figure(figsize=(14, 6))` before the call gives room to read.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
plt.figure(figsize=(14, 6))
sklearn.tree.plot_tree(
  ...,
  max_depth=...,
  feature_names=...,
  filled=True,
  fontsize=10,
)
plt.show()
print(list(importances.index[:5]))
```

### Solution

In [ ]:
plt.figure(figsize=(14, 6))
sklearn.tree.plot_tree(
  rfr.estimators_[0],
  max_depth=2,
  feature_names=list(train_X.columns),
  filled=True,
  fontsize=10,
)
plt.show()
print("The 5 most important variables:", list(importances.index[:5]))

Not necessarily. In the solution, the root splits on `GarageArea`, the fifth variable by importance, and `OverallQual`, the most important one, only shows up at the next level. At each split, the tree only chooses among a third of the variables, drawn at random (`max_features=0.33`), and it's trained on its own sample of rows: that's what decorrelates the forest's trees. The impurity-based importance, averaged over all the trees, smooths out these random draws.

## Out-of-bag (OOB) error

Each tree of the forest is trained on a draw with replacement of the rows: about 37% of the houses (more if `max_samples` is below 1) aren't in it, and are *out-of-bag* for this tree. A forest's OOB error predicts each house with only the trees that didn't see it: it's a free validation, with no cross-validation (`oob_score=True`).

*Plot the OOB error (an RMSE) of a copy of `rfr` as a function of its number of trees, from 25 to 300 in steps of 25, with the search's cross-validation RMSE as a reference. From how many trees on does the error level off? Is it close to the cross-validation RMSE?*

**Useful documentation**

- [`sklearn.base.clone`](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html)
- [`sklearn.metrics.root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)
- [`warm_start` in scikit-learn](https://scikit-learn.org/stable/glossary.html#term-warm_start)

In [ ]:
# Your code here

### Additional hints

- `sklearn.base.clone(rfr).set_params(warm_start=True, oob_score=True)`: an untrained forest with the same parameters as `rfr`, which stays intact for what follows.
- With `warm_start=True`, a `fit` after `set_params(n_estimators=n)` only adds the trees missing to reach `n`, instead of retraining the whole forest (as in the `random-forest-basics` demo).
- `forest.oob_prediction_`: the OOB prediction of each house of `train_X`; `sklearn.metrics.root_mean_squared_error(train_y, forest.oob_prediction_)` gives its RMSE. `forest.oob_score_`, on the other hand, is the R² of these predictions, not an error.
- `plt.axhline(-rscv.best_score_, linestyle="--")` draws the horizontal reference.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
forest = sklearn.base.clone(rfr).set_params(warm_start=..., oob_score=...)
n_trees = range(25, 301, 25)
oob_rmse = []
for n in n_trees:
  forest.set_params(n_estimators=n)
  forest.fit(...)
  oob_rmse.append(sklearn.metrics.root_mean_squared_error(train_y, ...))

plt.plot(n_trees, oob_rmse, label="OOB error")
plt.axhline(..., color="gray", linestyle="--", label="Cross-validation")
plt.legend()
plt.show()
```

### Solution

In [ ]:
# An untrained copy of rfr, which stays intact
forest = sklearn.base.clone(rfr).set_params(warm_start=True, oob_score=True)
n_trees = range(25, 301, 25)
oob_rmse = []
for n in n_trees:
  # warm_start=True: fit only adds the missing trees
  forest.set_params(n_estimators=n)
  forest.fit(train_X, train_y)
  oob_rmse.append(
    sklearn.metrics.root_mean_squared_error(train_y, forest.oob_prediction_)
  )

plt.plot(n_trees, oob_rmse, label="OOB error")
plt.axhline(
  -rscv.best_score_,
  color="gray",
  linestyle="--",
  label="Cross-validation RMSE (search)",
)
plt.xlabel("Number of trees")
plt.ylabel("RMSE ($)")
plt.title("OOB error as a function of the number of trees")
plt.legend()
plt.show()

The OOB error drops quickly up to about a hundred trees, then levels off around $29,700: beyond 100 to 150 trees, adding more only costs time. It stays a little above the cross-validation RMSE ($28,300), because each house is only predicted by the trees that didn't draw it, about half of the forest here. The two estimates stay close, and the OOB error comes from a single training, instead of 5.

## Evaluation on the validation set

The previous RMSEs were used to choose the model: the best of the search's 20 configurations is a little optimistic. The validation set, held out at the start, wasn't used for any choice.

*Compute the RMSE of `rfr` on the validation set (`val_X`, `val_y`), and compare it with the cross-validation RMSE and with the OOB error.*

**Useful documentation**

- [`sklearn.metrics.root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html)

In [ ]:
# Your code here

### Additional hints

- `rfr.predict(val_X)`: the predicted prices of the validation houses.
- `sklearn.metrics.root_mean_squared_error(val_y, predictions)`: their RMSE, in dollars.
- `-rscv.best_score_` and `oob_rmse[-1]`: the two RMSEs to compare.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
val_rmse = sklearn.metrics.root_mean_squared_error(..., rfr.predict(...))
print(f"RMSE on the validation set: ${val_rmse:,.0f}")
print(f"Cross-validation RMSE: ${...:,.0f}")
print(f"OOB error (300 trees): ${...:,.0f}")
```

### Solution

In [ ]:
val_rmse = sklearn.metrics.root_mean_squared_error(val_y, rfr.predict(val_X))
print(f"RMSE on the validation set: ${val_rmse:,.0f}")
print(f"Cross-validation RMSE: ${-rscv.best_score_:,.0f}")
print(f"OOB error (300 trees): ${oob_rmse[-1]:,.0f}")

The RMSE on the validation set (about $34,400) is higher than the cross-validation one ($28,300), but stays within the spread from one fold to another (from $20,000 to $36,000 depending on the fold). On 292 houses, a few large errors weigh a lot in an RMSE: the two largest ones here come from two houses of more than 4,000 sq ft, those that the exploration suggests removing from the training data; without them, the validation RMSE would drop to about $25,700. We don't remove them from the validation set though: in production, we don't choose the houses to estimate. This estimate, on data used for no choice, is the one we announce for the model.

## Conclusion

- Averaging decorrelated trees reduces the variance of a single tree: the forest makes much smaller errors than a tree, with almost no tuning.
- Tuning by random search then brings little (2% here).
- The forest also gives an estimate of its error (OOB) and an importance of the variables.
- A set held out from the start, used for no choice, gives the error to expect on new houses.

The next cell compares the tuned forest with boosted trees (gradient boosting), here scikit-learn's `HistGradientBoostingRegressor`, with no tuning at all: on tabular data, it's today's reference model, often a little better than a forest.

In [ ]:
hgb_rmse = score(sklearn.ensemble.HistGradientBoostingRegressor(random_state=0))
print(f"HistGradientBoostingRegressor: ${hgb_rmse[0]:,.0f} ± ${hgb_rmse[1]:,.0f}")
print(f"Tuned forest: ${-rscv.best_score_:,.0f}")